# 11 · Quantos *tokens* têm os textos?

Tarefa E5-T1 do [`docs/backlog.md`](../docs/backlog.md). Antes de tunar o BERTimbau e o Albertina é preciso escolher o **comprimento máximo**: o número de *tokens* a partir do qual o texto é cortado. Um *token* é um pedaço de texto que o modelo conhece. Uma palavra comum pode ser um *token* ("Câmara"); uma palavra rara parte-se em vários ("Aquisição" → `Aqui` `##si` `##ção`). Cada modelo tem o seu tokenizador, com o seu vocabulário, e por isso o mesmo texto pode ter números de *tokens* diferentes nos dois.

Porque importa: o comprimento máximo limita o tamanho de cada lote. Se todos os textos forem enchidos até ao máximo, 256 custa o dobro de 128 ou mais; se cada lote só for até ao seu texto mais comprido (*padding* dinâmico), como quase todos os textos são curtos, custa muito menos. Não se mede aqui: mede-se no E5-T4. Mas um texto cortado perde o que estava no fim.

**Regra, escrita no backlog antes de medir** (e em `src/bert_models.py`): para cada modelo, o comprimento máximo é o menor de 128, 256 e 512 em que pelo menos 99% dos textos de **treino** cabem inteiros, contando os *tokens* especiais que o modelo junta a cada texto. Se nem 512 chegar, fica 512 e diz-se quanto se corta. A validação mostra-se só como informação.

Não usa GPU nem os pesos dos modelos: só os tokenizadores, descarregados do Hugging Face na versão fixa de `src/bert_models.py`.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import transformers
from transformers import AutoConfig, AutoTokenizer

sys.path.insert(0, str(Path.cwd().parent / "src"))
import bert_models as bm

RAIZ = Path.cwd().parent
DATA = RAIZ / "data" / "processed"
# Textos com mais de 512 tokens dão um aviso por texto; aqui só se contam, não se dão ao modelo.
transformers.logging.set_verbosity_error()
print("transformers", transformers.__version__)
pd.DataFrame(bm.MODELOS).T

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


transformers 5.17.0


,nome,revisao
bertimbau,neuralmind/bert-base-portuguese-cased,94d69c95f98f7d5b2a8700c420230ae10def0baa
albertina,PORTULAN/albertina-100m-portuguese-ptpt-encoder,d15e086005cdab26d746d9113984cc654439df2b


## Os textos

O texto de entrada é a coluna `texto` do `prepare.py` (objeto + descrição + acordo-quadro, separados por `" | "`), tal como está: com maiúsculas e acentos, sem a normalização do TF-IDF. É este texto que o BERT vai ler.

No treino conta-se **uma linha por texto** do `treino.parquet` (o `prepare.py` já juntou os textos repetidos), porque é isso que o modelo vê em cada época. Ainda não se aplica a regra dos códigos raros: o N do BERT só se escolhe no E5-S1, e ela tira poucos exemplos.

In [2]:
treino = pd.read_parquet(DATA / "treino.parquet", columns=["texto"])["texto"]
val = pd.read_parquet(DATA / "validacao.parquet", columns=["texto"])["texto"]
assert treino.notna().all() and val.notna().all()
print(f"treino: {len(treino):,} textos; validação: {len(val):,} contratos".replace(",", " "))

treino: 478 384 textos; validação: 23 945 contratos


## Os dois tokenizadores

Cada um é carregado na versão fixa (`revisao`). A célula mostra quantos *tokens* especiais cada modelo junta a um texto (`[CLS]` no início, `[SEP]` no fim) e como parte uma frase inventada.

Nota: o Albertina usa o tokenizador **original do DeBERTa, feito para inglês** (diz o cartão do modelo), por isso as palavras portuguesas com acentos partem-se em mais pedaços. Os pedaços estranhos como `Ã§` são só a forma como este tipo de tokenizador (BPE ao nível do byte) mostra o "ç": não é um erro.

In [3]:
tok = {m: AutoTokenizer.from_pretrained(v["nome"], revision=v["revisao"]) for m, v in bm.MODELOS.items()}

EXEMPLO = "Aquisição de serviços de manutenção de equipamentos informáticos | Câmara Municipal de Évora"
ESPECIAIS = {}
for m, t in tok.items():
    ESPECIAIS[m] = len(t("")["input_ids"])
    ids = t(EXEMPLO)["input_ids"]
    print(f"{m}: {ESPECIAIS[m]} tokens especiais; frase de exemplo = {len(ids)} tokens")
    print("   ", t.convert_ids_to_tokens(ids))
assert all(n == 2 for n in ESPECIAIS.values())

# Quantas posições aceita cada modelo (o comprimento máximo não pode passar daqui).
for m, v in bm.MODELOS.items():
    cfg = AutoConfig.from_pretrained(v["nome"], revision=v["revisao"])
    print(f"{m}: {cfg.model_type}, max_position_embeddings = {cfg.max_position_embeddings}")
    assert cfg.max_position_embeddings >= max(bm.CANDIDATOS)

bertimbau: 2 tokens especiais; frase de exemplo = 18 tokens
    ['[CLS]', 'Aqui', '##si', '##ção', 'de', 'serviços', 'de', 'manutenção', 'de', 'equipamentos', 'inform', '##áticos', '|', 'Câmara', 'Municipal', 'de', 'Évora', '[SEP]']
albertina: 2 tokens especiais; frase de exemplo = 35 tokens
    ['[CLS]', 'A', 'qu', 'isi', 'Ã§', 'Ã£o', 'Ġde', 'Ġserv', 'i', 'Ã§', 'os', 'Ġde', 'Ġman', 'uten', 'Ã§', 'Ã£o', 'Ġde', 'Ġequip', 'ament', 'os', 'Ġinform', 'Ã¡', 'tic', 'os', 'Ġ|', 'ĠC', 'Ã¢', 'm', 'ara', 'ĠMunicipal', 'Ġde', 'ĠÃī', 'v', 'ora', '[SEP]']
bertimbau: bert, max_position_embeddings = 512
albertina: deberta, max_position_embeddings = 512


## Contar

`comprimento` é o número de *tokens* de cada texto, já com os especiais, sem cortar nada. Corre aos blocos de 20 000 textos para não guardar todos os *tokens* em memória (só o comprimento). Demora alguns minutos no portátil.

In [4]:
def comprimentos(t, textos, bloco=20_000):
    out = np.empty(len(textos), dtype=np.int32)
    lista = textos.tolist()
    for i in range(0, len(lista), bloco):
        ids = t(lista[i:i + bloco], add_special_tokens=True, truncation=False)["input_ids"]
        out[i:i + len(ids)] = [len(x) for x in ids]
    return out


# A função conta os tokens especiais: na frase de exemplo dá o mesmo que a célula de cima.
for m, t in tok.items():
    assert comprimentos(t, pd.Series([EXEMPLO]))[0] == len(t(EXEMPLO)["input_ids"]) > ESPECIAIS[m]

comp = {(m, conj): comprimentos(t, textos)
        for m, t in tok.items() for conj, textos in [("treino", treino), ("validação", val)]}
palavras = {conj: textos.str.split().str.len().to_numpy() for conj, textos in [("treino", treino), ("validação", val)]}

## Quantos textos cabem inteiros

Para cada modelo e conjunto: a percentagem de textos com comprimento ≤ 128, 256 e 512, e alguns percentis (o percentil 99 é o comprimento que 99% dos textos não passam). Para comparar, a última coluna dá a mediana em palavras (separadas por espaços).

In [5]:
linhas = []
for (m, conj), n in comp.items():
    linha = {"modelo": m, "conjunto": conj}
    for L in bm.CANDIDATOS:
        linha[f"≤ {L}"] = 100 * (n <= L).mean()
    for p in (50, 90, 99):
        linha[f"p{p}"] = np.percentile(n, p)
    linha["máximo"] = n.max()
    linha["tokens por palavra (mediana)"] = np.median(n / np.maximum(palavras[conj], 1))
    linha["mediana em palavras"] = np.median(palavras[conj])
    linhas.append(linha)
tabela = pd.DataFrame(linhas).set_index(["modelo", "conjunto"])
tabela.round(2)

≤ 128  ≤ 256  ≤ 512   p50    p90    p99  máximo  \
modelo    conjunto                                                     
bertimbau treino     98.17  99.81  99.98  31.0   72.0  154.0    5713   
          validação  94.22  98.10  99.91  33.0   94.0  316.0     851   
albertina treino     96.21  99.63  99.96  40.0   90.0  190.0    7844   
          validação  91.84  97.81  99.92  42.0  115.0  372.0    1009   

                     tokens por palavra (mediana)  mediana em palavras  
modelo    conjunto                                                      
bertimbau treino                             2.05                 15.0  
          validação                          2.12                 15.0  
albertina treino                             2.72                 15.0  
          validação                          2.79                 15.0

## Os textos mais compridos

Quantos textos de treino passam cada limite, em número, e um exemplo do que fica para lá de 512 *tokens* (o mais comprido no BERTimbau, só o início).

In [6]:
for m in tok:
    n = comp[(m, "treino")]
    print(m, {f"> {L}": int((n > L).sum()) for L in bm.CANDIDATOS})
i = int(np.argmax(comp[("bertimbau", "treino")]))
print("\nmais comprido:", comp[("bertimbau", "treino")][i], "tokens;", treino.iloc[i][:400], "...")

bertimbau {'> 128': 8760, '> 256': 890, '> 512': 83}
albertina {'> 128': 18151, '> 256': 1786, '> 512': 186}

mais comprido: 5713 tokens; Aquisição de serviços de Seguros Diversos, por lotes – para o ano Letivo 2024/2025: Designação: Lote 1 - Acidentes de Trabalho, Acidentes Pessoais - Alunos, Responsabilidade Civil Alunos em FCT Modalidade: Seguro de Acidentes de Trabalho – pagamento mensal Prémio variável (folha de férias men-sal) Todas as constantes das Folhas de Férias a enviar men-salmente à Seguradora Responsabilidade emergent ...


## Escolha, pela regra escrita antes de medir

Usa só o treino. A célula para com erro se `COMPRIMENTO_MAXIMO` em `src/bert_models.py` (o valor que o treino vai ler) não for o que a regra escolhe aqui.

In [7]:
escolha = {}
for m in tok:
    n = comp[(m, "treino")]
    cabe = [L for L in bm.CANDIDATOS if (n <= L).mean() >= bm.QUOTA]
    escolha[m] = cabe[0] if cabe else max(bm.CANDIDATOS)
    print(f"{m}: {escolha[m]} tokens; ficam inteiros {100 * (n <= escolha[m]).mean():.2f}% do treino "
          f"e {100 * (comp[(m, 'validação')] <= escolha[m]).mean():.2f}% da validação; "
          f"cortados {int((n > escolha[m]).sum())} textos de treino")
assert bm.COMPRIMENTO_MAXIMO == escolha, f"atualizar src/bert_models.py: {escolha}"

bertimbau: 256 tokens; ficam inteiros 99.81% do treino e 98.10% da validação; cortados 890 textos de treino
albertina: 256 tokens; ficam inteiros 99.63% do treino e 97.81% da validação; cortados 1786 textos de treino
